In [9]:
%pip install -q transformers datasets evaluate rouge_score sentencepiece accelerate nltk sumy pyarrow pandas matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
import random
import warnings

import torch
import numpy as np
import pandas as pd
import nltk
import evaluate

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

warnings.filterwarnings("ignore")

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070


In [22]:
from datasets import load_dataset, DatasetDict

raw_dataset = load_dataset(
    "parquet",
    data_files={
        "train": "xlsum-train.parquet",
        "validation": "xlsum-validation.parquet",
        "test": "xlsum-test.parquet",
    }
)

train_data = raw_dataset["train"].shuffle(seed=SEED).select(range(3000))
val_data = raw_dataset["validation"].shuffle(seed=SEED).select(range(500))
test_data = raw_dataset["test"].shuffle(seed=SEED).select(range(500))

dataset = DatasetDict({
    "train": train_data,
    "validation": val_data,
    "test": test_data
})

print(dataset)
print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 3000
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 500
    })
})
Train: 3000
Validation: 500
Test: 500


In [23]:
MODEL_NAME = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.to("cuda" if torch.cuda.is_available() else "cpu")

print(model.config._name_or_path)
print(tokenizer.__class__)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

facebook/bart-base
<class 'transformers.models.roberta.tokenization_roberta.RobertaTokenizer'>


In [24]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 64

def preprocess_function(examples):
    inputs = examples["text"]
    targets = examples["summary"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_train = tokenized_dataset["train"]
tokenized_val = tokenized_dataset["validation"]
tokenized_test = tokenized_dataset["test"]

print(tokenized_dataset)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
})


In [25]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

In [26]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.asarray(predictions)
    labels = np.asarray(labels)

    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)
    labels = np.where(labels == -100, tokenizer.pad_token_id, labels)

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {key: round(value * 100, 4) for key, value in result.items()}

In [27]:
model.gradient_checkpointing_enable()
model.config.use_cache = False

model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-xlsum-indonesia-3000",

    eval_strategy="epoch",
    save_strategy="no",

    learning_rate=3e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    weight_decay=0.01,

    num_train_epochs=3,

    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=4,

    fp16=torch.cuda.is_available(),

    logging_steps=50,
    report_to="none",

    load_best_model_at_end=False
)

In [28]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [29]:
small_eval = tokenized_val.select(range(5))
trainer.evaluate(eval_dataset=small_eval)

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
No log,4.605780,0,16.835000,2.962000,9.273300,9.270900


{'eval_loss': 4.605780124664307,
 'eval_rouge1': 16.835,
 'eval_rouge2': 2.962,
 'eval_rougeL': 9.2733,
 'eval_rougeLsum': 9.2709}

In [31]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,12.207379,2.737541,21.037700,6.529500,16.960800,16.959800
2,11.107570,2.588922,21.830600,7.133600,17.713200,17.669200
3,10.519374,2.539148,22.753400,7.719600,18.690500,18.673600


In [32]:
test_results = trainer.evaluate(eval_dataset=tokenized_test)

print("Test Results:")
print(test_results)

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
10.519374,2.556399,3,23.267900,7.865900,18.637800,18.652300


Test Results:
{'eval_loss': 2.556398630142212, 'eval_rouge1': 23.2679, 'eval_rouge2': 7.8659, 'eval_rougeL': 18.6378, 'eval_rougeLsum': 18.6523}


In [33]:
save_path = "./bart-xlsum-indonesia-final"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./bart-xlsum-indonesia-final


In [34]:
def summarize_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)

    summary_ids = model.generate(
        **inputs,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

sample_text = dataset["test"][0]["text"]
reference_summary = dataset["test"][0]["summary"]
generated_summary = summarize_text(sample_text)

print("ARTICLE:")
print(sample_text[:1000])

print("\nREFERENCE SUMMARY:")
print(reference_summary)

print("\nGENERATED SUMMARY:")
print(generated_summary)

ARTICLE:
Pandemi Covid-19 telah meningkatkan minat berinvestasi emas. Kenaikan harga itu salah satu penyebab utamanya karena permainan para pedagang emas dan permintaan yang tinggi, tapi di balik itu, muncul pertanyaan tentang berapa sisa pasokan logam mulia itu di bumi dan kapan akan habis. Emas menjadi buruan masyarakat karena dapat dijadikan sebagai investasi, simbol status ekonomi, dan komponen utama produk elektronik. Tapi jumlah emas di dunia terbatas, dan pada akhirnya akan datang satu saat ketika tidak ada lagi emas yang tersisa untuk ditambang. Puncak produksi emas Beberapa ahli berbicara tentang apakah dunia telah mencapai puncak produksi emas - diukur dengan jumlah terbanyak emas yang pernah ditambang dalam satu tahun - atau tidak. Hal itu ditunjukan dengan mulai menurunnya tren produksi emas dunia. Contohnya, pada tahun 2019, produksi tambang emas dunia turun 1% menjadi 3.531 ton dibandingkan tahun 2018, menurut Dewan Emas Dunia. Penurunan produksi tahunan ini yang pertama 